# EchoPilot Research 03: QLoRA Fine-Tuning of Qwen2.5-3B-Instruct

This notebook details the supervised fine-tuning (SFT) pipeline to turn a base local instruct model
into a reliable structured MCQ generator:
- **Base Model**: `Qwen/Qwen2.5-3B-Instruct` (real weights, Apache 2.0 license, fits in 4-bit VRAM/RAM).
- **Quantization**: 4-bit NormalFloat (NF4) with double quantization via `bitsandbytes`.
- **PEFT / LoRA**: Low-Rank Adaptation targeting attention and MLP projection layers (`q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`).
- **Target Schema**: Strict JSON output with prompt, distinct options, evidence span, and answer index.


In [ ]:
import os
import sys
import json
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available:  {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("Running in CPU mode. (Fine-tuning is designed for CUDA; model inference can run on CPU).")


## Step 1: Format SFT Training Data (ChatML Format)
We format our reviewed items (`../data/sample_training_data.jsonl`) into conversation messages compatible with Qwen's ChatML template:
`<|im_start|>system...<|im_end|><|im_start|>user...<|im_end|><|im_start|>assistant...<|im_end|>`.


In [ ]:
data_file = "../data/sample_training_data.jsonl"
training_records = []

with open(data_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        system_msg = "You are an expert pedagogical quiz generator. Output ONLY a valid JSON object matching the requested schema."
        user_prompt = f"Topic: {item['skill']}\nDifficulty: {item['difficulty']}\nEvidence: {item['evidence_quote']}\nGenerate a single 4-option multiple-choice question."
        assistant_resp = json.dumps({
            "topic": item["skill"],
            "question": item["question"],
            "options": item["options"],
            "answer": item["answer"],
            "explanation": item["explanation"],
            "difficulty": item["difficulty"]
        }, indent=2)
        
        training_records.append({
            "messages": [
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": assistant_resp}
            ]
        })

print(f"Loaded {len(training_records)} training conversation records.")
print("\nExample conversation:")
print(json.dumps(training_records[0], indent=2))


## Step 2: Configure QLoRA BitsAndBytes & LoRA Adapters
Here is the configuration for 4-bit fine-tuning.


In [ ]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, TaskType

# 4-bit Quantization Config (for CUDA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
)

# LoRA Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print("LoRA Config:")
print(lora_config)


## Step 3: SFT Training Pipeline Specification
The code below initializes the model and sets up Hugging Face `TRL`'s `SFTTrainer`.


In [ ]:
script_template = """
# Command to execute training on a CUDA-equipped machine:
# python -m trl.commands.sft \
#     --model_name_or_path "Qwen/Qwen2.5-3B-Instruct" \
#     --dataset_name "../data/sample_training_data.jsonl" \
#     --output_dir "../models/qwen2.5_3b_quiz_lora" \
#     --per_device_train_batch_size 2 \
#     --gradient_accumulation_steps 4 \
#     --learning_rate 2e-4 \
#     --num_train_epochs 3 \
#     --fp16 \
#     --logging_steps 10 \
#     --save_strategy "epoch"
"""
print("Training execution script ready.")
print(script_template)


## Step 4: Output Validation & Schema Conformance Check
We test the generator's JSON parsing and schema validation using `validate_and_parse_quiz_json`.


In [ ]:
sys.path.append(os.path.abspath("../src"))
from generator import validate_and_parse_quiz_json

sample_model_output = """
```json
{
  "title": "Operating Systems Deadlock Essentials",
  "questions": [
    {
      "topic": "Deadlock Conditions",
      "question": "Which condition requires that a process hold at least one resource while waiting for others?",
      "options": ["Mutual exclusion", "Hold and wait", "No preemption", "Circular wait"],
      "answer": 1,
      "explanation": "Hold and wait specifically describes retaining existing resources while requesting additional ones.",
      "difficulty": "steady"
    }
  ]
}
```
"""

parsed = validate_and_parse_quiz_json(sample_model_output)
print("Schema Validation Passed!")
print(f"Quiz Title: {parsed['title']}")
print(f"Num Questions: {len(parsed['questions'])}")
print(f"Q1 Answer: Option {parsed['questions'][0]['answer']} ({parsed['questions'][0]['options'][parsed['questions'][0]['answer']]})")
